<a href="https://colab.research.google.com/github/lodigasatish-ai/teleconnect-ml-assignment/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Task 2: Data Preprocessing
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# dataset
print("Loading dataset...")
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
print("Original shape:", df.shape)

Loading dataset...
Original shape: (7043, 21)


In [2]:
# Basic cleaning
print("\nCleaning data...")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
df.drop(columns=['customerID'], inplace=True)
print("After cleaning:", df.shape)


Cleaning data...
After cleaning: (7032, 20)


In [3]:
# Encoding
print("\nEncoding categorical data...")
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
cat_cols.remove('Churn')
le = LabelEncoder()
df['Churn'] = le.fit_transform(df['Churn'])
df = pd.get_dummies(df, columns=cat_cols)
print("Encoding completed.")


Encoding categorical data...
Encoding completed.


In [4]:
#Feature Engineering
print("\nCreating new features...")
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)
service_cols = [col for col in df.columns if 'Yes' in col]
df['ServiceCount'] = df[service_cols].sum(axis=1)
df['RemainingMonths'] = 12 - (df['tenure'] % 12)
df['ContractValue'] = df['MonthlyCharges'] * df['RemainingMonths']
print("New features added.")


Creating new features...
New features added.


In [5]:
# Scaling
print("\nApplying scaling...")
num_cols = ['tenure','MonthlyCharges','TotalCharges','AvgMonthlySpend','ContractValue']

std_scaler = StandardScaler()
df_std = df.copy()
df_std[num_cols] = std_scaler.fit_transform(df[num_cols])

mm_scaler = MinMaxScaler()
df_mm = df.copy()
df_mm[num_cols] = mm_scaler.fit_transform(df[num_cols])
print("Scaling done using StandardScaler and MinMaxScaler.")


Applying scaling...
Scaling done using StandardScaler and MinMaxScaler.


In [6]:
#Handle imbalance
print("\nHandling class imbalance...")
X = df_std.drop('Churn', axis=1)
y = df_std['Churn']

smote = SMOTE(random_state=42)
X_sm, y_sm = smote.fit_resample(X, y)
print("After SMOTE:", X_sm.shape)

rus = RandomUnderSampler(random_state=42)
X_under, y_under = rus.fit_resample(X, y)
print("After Undersampling:", X_under.shape)


Handling class imbalance...
After SMOTE: (10326, 49)
After Undersampling: (3738, 49)


In [7]:
# Feature selection
print("\nSelecting important features...")

mi_scores = mutual_info_classif(X_sm, y_sm)
mi_results = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
print("\nTop 5 features (Mutual Information):")
print(mi_results.head(5))

rfe = RFE(RandomForestClassifier(n_estimators=50, random_state=42), n_features_to_select=10)
rfe.fit(X_sm, y_sm)
rfe_features = X.columns[rfe.support_]
print("\nTop features selected by RFE:")
print(list(rfe_features))


Selecting important features...

Top 5 features (Mutual Information):
tenure                     0.237884
Contract_Month-to-month    0.151683
MonthlyCharges             0.131221
TechSupport_No             0.110366
OnlineSecurity_No          0.102689
dtype: float64

Top features selected by RFE:
['tenure', 'MonthlyCharges', 'TotalCharges', 'OnlineSecurity_No', 'TechSupport_No', 'Contract_Month-to-month', 'PaymentMethod_Electronic check', 'AvgMonthlySpend', 'ServiceCount', 'ContractValue']


In [8]:
#Train / Validation / Test split
print("\nSplitting dataset...")
X_train, X_temp, y_train, y_temp = train_test_split(
    X_sm, y_sm, test_size=0.30, stratify=y_sm, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("\nFinal split:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nTask 2 completed successfully. Ready for model building.")


Splitting dataset...

Final split:
Train: (7228, 49)
Validation: (1549, 49)
Test: (1549, 49)

Task 2 completed successfully. Ready for model building.
